<a href="https://colab.research.google.com/github/hand-e-fr/OpenHosta/blob/doc/docs/openhosta_QwQ32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenHosta basic example with a Local Mistral-Small

This colab demonstrate simple use cases of OpenHosta. If you do not have an OpenAI (or other) API KEY, you can run the first part **Install a local Mistral-Small**. Otherwise, directly jump to step 2: **Basic Usage**.

## Install a local model (qwen-coder or Mistral-Small)

In [1]:
!apt install -y screen

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  byobu | screenie | iselect ncurses-term
The following NEW packages will be installed:
  screen
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 672 kB of archives.
After this operation, 1,029 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 screen amd64 4.9.0-1 [672 kB]
Fetched 672 kB in 1s (704 kB/s)
Selecting previously unselected package screen.
(Reading database ... 124947 files and directories currently installed.)
Preparing to unpack .../screen_4.9.0-1_amd64.deb ...
Unpacking screen (4.9.0-1) ...
Setting up screen (4.9.0-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!screen -dmS ollama ollama serve

In [5]:
#!ollama run qwen2.5-coder:3b hello --verbose  2>&1 | grep -E ":"
!ollama run hf.co/bartowski/Qwen_QwQ-32B-GGUF:Q4_K_S hello --verbose  2>&1 | grep -E "([^0]0%|Bonjour|:)"
# Can take 5 minutes
#!ollama run hf.co/bartowski/Mistral-Small-24B-Instruct-2501-GGUF:Q4_K_S hello --verbose  2>&1 | grep -E "([^0]0%|Bonjour|:)"

pulling dcb3460517e6...   0% ▕                ▏    0 B/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  59 KB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏ 149 KB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏ 1.4 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  10 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  17 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  30 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  44 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  50 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕                ▏  64 MB/ 18 GB                  pulling manifest 
pulling dcb3460517e6...   0% ▕

In [6]:
!pip install OpenHosta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.4 MB/s eta 0:00:00


## Basic Usage of OpenHosta

### Configure the LLM that you want to use

In [17]:
from OpenHosta import config

# Use Mistral-Small-24B (2025 Jannuary) through ollama
my_model=config.Model(
    base_url="http://localhost:11434/v1/chat/completions",
    json_output=False,
    model="hf.co/bartowski/Qwen_QwQ-32B-GGUF:Q4_K_S", api_key="none", timeout=120
)
config.set_default_model(my_model)


### Emulate functions using the seleted LLM

In [9]:
from OpenHosta.asynchrone import emulate

In [18]:
async def translate(text:str, language:str)->str:
    """
    This function translates the text in the “text” parameter into the language specified in the “language” parameter.
    """
    return await emulate()

result = await translate("Hello World!", "French")
print(result)

Bonjour le monde!


In [19]:
async def find_name_age(sentence:str, id:dict)->dict:
    """
    This function find in a text the name and the age of a personn.

    Args:
        sentence: The text in which to search for information
        id: The dictionary to fill in with information

    Return:
        A dictionary identical to the one passed in parameter, but filled with the information.
        If the information is not found, fill with the values with “None”.
    """
    return await emulate(model=my_model)

return_dict = {"name": "", "age": ""}
result = await find_name_age("Hello, I'm John Wick, i'm 30 and I live in New York", return_dict)
print(result)

{'name': 'John Wick', 'age': '30'}


### Specify advanced return types

In [27]:
from typing import Dict, Tuple, List

async def analyze_text(text: str) -> Dict[str, List[Tuple[int, str]]]:
    """Analyze text to map each word to a list of tuples containing word length and word."""
    return await emulate()

# Example usage
analysis = await analyze_text("Hello, World!")

print(analysis)
print(type(analysis))

{}
<class 'dict'>


[Model.response_parser] JSONDecodeError: Expecting ',' delimiter: line 1 column 58 (char 57)
Continuing the process.

In [28]:
from OpenHosta import print_last_response

print_last_response(analyze_text)

[THINKING]
Okay, let me try to figure out how to handle this function call. The user is asking me to emulate the analyze_text function which takes a string and returns a dictionary where each word maps to a list of tuples containing the word's length and the word itself.

First, I need to understand what the function is supposed to do. Each word in the input text should be split up, then for each word, calculate its length and pair that with the word as a tuple. Then these tuples are collected into lists under each corresponding word key in a dictionary. But wait, how does the mapping work? Maybe the keys are unique words from the text, right?

Let me look at the example input: analyze_text(text='Hello, World!'). The sentence has two words: "Hello," and "World!". Wait, but actually punctuation might be attached here. Should they be considered part of the word or stripped out before processing? Hmm. Because the function is analyzing text by breaking into 'words', I need to decide how 'w

### Specify pydantic return strucures

OpenHosta is compatible with pydantic. You can specify pydantic input and output types and OpenHosta will propagate schema and Field documentation to the LLM.

In [21]:
!pip install pydantic

In [24]:
from pydantic import BaseModel, Field

class Personn(BaseModel):
    name: str = Field(..., description = "The full name")
    age: int

async def find_name_age_pydantic(sentence:str)->Personn:
    """
    This function find in a text the name and the age of a personn.

    Args:
        sentence: The text in which to search for information

    Return:
        A Pydantic model, but filled with the information.
        If the information is not found, fill with the values with “None”.
    """
    return await emulate()

result = await find_name_age_pydantic("Luke Skywalker is very surprising: he's only 27 when he becomes a Jedi.")
print(result)

name='Luke Skywalker' age=27


In [25]:
from OpenHosta import print_last_response

print_last_response(find_name_age_pydantic)

[THINKING]
Okay, let me see. The user wants me to find the name and age in the given sentence.

First, looking at the sentence provided: "Luke Skywalker is very surprising: he's only 27 when he becomes a Jedi." 

Hmm, the name here seems straightforward—it’s Luke Skywalker. That part looks clear because it's presented right after "is" and before the colon. 

Now for the age. The sentence mentions '27' which is a number. Since age is an integer, that should be 27. But I need to make sure there are no other numbers or possible confusion. The phrase says "only 27 when he becomes a Jedi," so it's pretty explicit about his age at that point.

The Pydantic model requires both name and age. So the name is "Luke Skywalker" and age is 27. 

Wait, check if there's any reason to doubt this? Maybe the structure of the sentence could mislead? Like "he's" referring back to Luke. The sentence structure clearly connects the age to him. Also, the task says to fill with None if not found, but here both 

### Limitations

The emulation is limited by the LLM capabilities. Try to have it count r in strawberrry and you will get into troubles ;-).
Make sure the LLM is capable and not alucinating before you implement an emulated function.

In [26]:
async def find_occurence_of_a_word(word :str, text: str) -> int:
    """
    This function takes a word and a text and returns
    the number of times the word appears in the text.
    """
    return await emulate()

await find_occurence_of_a_word("Hello", "Hello World Hello!")


RequestError: [Model.api_call] Request failed:
HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)


